# Normalizing Entities

#### INPUT  
- narrative_extractions.csv

#### WHAT THIS NOTEBOOK DOES
- loads the raw extractions produced by notebook 03_extract.ipynb
- embeds the unique entity / narrative labels with a multilingual sentence model
- pass 1: clusters near-identical string variants via a kNN similarity graph
- LLM post-processing: adjudicates candidate merge pairs within each specific_type group
- LLM post-processing: attaches low-frequency singletons to established canonicals within each type group (iterative)
- splits over-merged canonicals and applies the resulting normalization map to every slot
- runs sanity checks on the normalized labels

#### OUTPUTS 
- narrative_extractions_normalized.csv --> extractions with canonical entity labels
- entity_normalization_map.csv --> raw label → canonical label mapping
- entity_label_embeddings.npy / entity_label_strings.npy --> cached label embeddings
- pass1_emb_norm.npy / pass1_ids.npy --> cached pass-1 embeddings


In [1]:
# imports

import numpy as np
import pandas as pd
from config import normalize_config as cfg
from lib.normalize_lib import (
    load_extractions, get_unique_labels,
    embeddings_exist, load_embeddings, save_embeddings, embed_labels,
    diagnostic, build_canonical_map,
    apply_normalization, save_outputs, sanity_check,
    build_canonical_type_map,
    find_split_candidates, llm_split_canonicals, apply_splits,
    find_candidate_pairs_by_type,
    find_singleton_attachment_pairs_by_type,
    llm_adjudicate_pairs_by_type,
    apply_llm_merges_by_type,
)
from prompts.normalize_prompt import SYSTEM_PROMPT as NORMALIZE_PROMPT

In [2]:
# load data

extractions = load_extractions(cfg.EXTRACTIONS_CSV)
label_counts = get_unique_labels(extractions)

print(f"Total rows:              {len(extractions):,}")
print(f"Unique labels:           {len(label_counts):,}")
print(f"Labels appearing once:   {(label_counts['frequency'] == 1).sum():,}")
print(f"Labels appearing ≥5:     {(label_counts['frequency'] >= 5).sum():,}")
print(f"Labels appearing ≥10:    {(label_counts['frequency'] >= 10).sum():,}")
print(f"\nTop 20 most frequent labels:")
print(label_counts.head(20).to_string(index=False))

Total rows:              417,583
Unique labels:           353,162
Labels appearing once:   333,830
Labels appearing ≥5:     3,114
Labels appearing ≥10:    1,083

Top 20 most frequent labels:
                               label  frequency
                        Donald Trump       1524
   Swiss Federal Council (Bundesrat)        754
                             Ukraine        632
                              Russia        612
                                Iran        462
                              Israel        444
United States (Trump administration)        431
           US President Donald Trump        398
         Federal Council (Bundesrat)        377
                       United States        355
                               Hamas        317
          SVP (Swiss People's Party)        284
           Swiss citizens and voters        268
Swiss Federal Government (Bundesrat)        267
    Donald Trump / US administration        257
Trump administration / US government     

In [3]:
# embed unique labels (and cache results to speed up future runs)

unique_labels = label_counts["label"].values

if embeddings_exist(cfg.ENTITY_EMBEDDINGS_NPY, cfg.ENTITY_LABELS_NPY):
    print("Loading cached label embeddings...")
    embeddings, labels = load_embeddings(cfg.ENTITY_EMBEDDINGS_NPY, cfg.ENTITY_LABELS_NPY)
else:
    print(f"Embedding {len(unique_labels):,} unique labels (this takes ~20-40 minutes)...")
    embeddings = embed_labels(unique_labels, cfg.EMBEDDING_MODEL, cfg.EMBEDDING_BATCH_SIZE)
    labels     = unique_labels
    save_embeddings(embeddings, labels, cfg.ENTITY_EMBEDDINGS_NPY, cfg.ENTITY_LABELS_NPY)

print(f"Embeddings shape: {embeddings.shape}")

Loading cached label embeddings...
Embeddings shape: (353162, 1024)


In [ ]:
# run diagnostics to understand the embedding space and set thresholds for clustering

diagnostic(
    embeddings=embeddings,
    labels=labels,
    label_counts=label_counts,
    thresholds=cfg.DIAGNOSTIC_THRESHOLDS,
    sample_size=cfg.DIAGNOSTIC_SAMPLE_SIZE,
    min_frequency=cfg.DIAGNOSTIC_MIN_FREQUENCY,
    show_top_n=cfg.DIAGNOSTIC_SHOW_TOP_N,
)

In [4]:
# run pass 1 and build canonical map
# pass 2 dropped — threshold 0.04 excludes opposite-framing pairs
# (e.g. "Passage of X" / "Rejection of X", distance ~0.04-0.05)
# LLM threshold lowered to 0.94 to catch the residual 0.04-0.05 range

import importlib
import lib.normalize_lib as normalize_lib
importlib.reload(normalize_lib)
from lib.normalize_lib import _pass1_connected_components, build_canonical_map
from sklearn.preprocessing import normalize as sk_normalize
from collections import Counter, defaultdict
from pathlib import Path

# load from cache if available, otherwise run pass 1
if Path(cfg.PASS1_EMB_NORM_NPY).exists() and Path(cfg.PASS1_IDS_NPY).exists():
    print("Loading pass 1 results from cache...")
    emb_norm  = np.load(cfg.PASS1_EMB_NORM_NPY)
    pass1_ids = np.load(cfg.PASS1_IDS_NPY)
    print("Loaded.")
else:
    print("Running pass 1 (faiss nearest neighbour graph, ~25 min)...")
    emb_norm  = sk_normalize(embeddings, norm="l2")
    pass1_ids = _pass1_connected_components(
        emb_norm, cfg.CLUSTER_THRESHOLD_PASS1, k=cfg.CLUSTER_KNN_K
    )
    np.save(cfg.PASS1_EMB_NORM_NPY, emb_norm)
    np.save(cfg.PASS1_IDS_NPY, pass1_ids)
    print("Saved pass 1 results to disk")

# break oversized components into singletons
cluster_ids = pass1_ids.copy().astype(int)
next_id = int(pass1_ids.max()) + 1
groups = defaultdict(list)
for i, cid in enumerate(pass1_ids):
    groups[int(cid)].append(i)
skipped = {cid: idxs for cid, idxs in groups.items()
           if len(idxs) > cfg.CLUSTER_MAX_COMPONENT_SIZE}
if skipped:
    print(f"Breaking {len(skipped):,} oversized components into singletons "
          f"(largest: {max(len(v) for v in skipped.values()):,} labels)...")
    for cid, idxs in skipped.items():
        for i in idxs:
            cluster_ids[i] = next_id
            next_id += 1

canonical_map = build_canonical_map(
    labels       = labels,
    cluster_ids  = cluster_ids,
    label_counts = label_counts,
)

size_counts = Counter(pass1_ids)
sizes = np.array(list(size_counts.values()))
print(f"\nTotal components: {len(sizes):,}")
print(f"Singletons (size=1): {(sizes == 1).sum():,}")
print(f"Size 2-5:   {((sizes >= 2) & (sizes <= 5)).sum():,}")
print(f"Size 6-20:  {((sizes >= 6) & (sizes <= 20)).sum():,}")
print(f"Size >100:  {(sizes > 100).sum():,}")
print(f"Largest component: {sizes.max():,}")
print(f"\nUnique labels before: {len(label_counts):,}")
print(f"Unique labels after:  {canonical_map['canonical_label'].nunique():,}")
print(f"Labels merged:        {(canonical_map['n_variants'] > 1).sum():,}")

Loading pass 1 results from cache...
Loaded.
Breaking 3 oversized components into singletons (largest: 20,603 labels)...

Total components: 283,033
Singletons (size=1): 263,597
Size 2-5:   17,759
Size 6-20:  1,443
Size >100:  25
Largest component: 20,603

Unique labels before: 353,162
Unique labels after:  305,054
Labels merged:        67,541


In [5]:
# sanity check

sanity_check(canonical_map, label_counts, n=50)

Top 50 normalized entities by number of variants:

  → European Union (EU institutions and member states) (21 occurrences)
      NATO allies (US, European states, Canada) (1)
      European NATO allies (Germany, Poland, Italy, United Kingdom) (1)
      NATO member states (Europe and North Atlantic region) (1)
      NATO (as institutional collective) (1)
      NATO allies (Germany, Finland, Denmark, Sweden, Norway, UK, Netherlands, Canada) (1)
      European states (Germany, France, UK, Poland, Italy, Finland, Norway, Denmark, Netherlands) (1)
      European NATO states (UK, France, Spain, Greece) (1)
      European Union (Commission and member states) (1)
      Baltic states (Estonia, Latvia) (1)
      Western allies (US, Europe) (1)
      NATO member states (Germany, Estonia, Poland) (1)
      European allies (NATO partners) (1)
      NATO member states (Poland, broader alliance) (1)
      Western allies (France, Britain, US) (1)
      NATO allies (Denmark, France, Germany, Poland) (1

In [6]:
n_above = (canonical_map["n_variants"] >= 20).sum()
n_groups = canonical_map[canonical_map["n_variants"] >= 20]["canonical_label"].nunique()
print(f"Labels with ≥20 variants: {n_above:,}")
print(f"Canonical groups with ≥20 variants: {n_groups:,}")

Labels with ≥20 variants: 11,560
Canonical groups with ≥20 variants: 251


In [7]:
# LLM split step: audit large canonical groups for pass-1 wrong merges

import importlib
import lib.normalize_lib as normalize_lib
importlib.reload(normalize_lib)
from lib.normalize_lib import find_split_candidates, llm_split_canonicals, apply_splits

split_candidates = find_split_candidates(
    canonical_map,
    label_counts,
    min_variants=20,
    top_n=300,
)

splits_by_canonical = llm_split_canonicals(split_candidates, label_counts)

# inspect before applying
print("\n=== SPLIT INSPECTION ===")
for canonical, splits in splits_by_canonical.items():
    print(f"\n[{canonical}] — {len(splits)} splits")
    for s in splits[:15]:
        print(f"  SPLIT: {s}")
    if len(splits) > 15:
        print(f"  ... and {len(splits)-15} more")

# only apply after inspecting — comment out if results look wrong
# if splits_by_canonical:
#     canonical_map = apply_splits(canonical_map, splits_by_canonical)

  251 canonical groups with ≥20 variants to audit


  [European Union (EU institutions and member states)] → 168 splits


  [Karin Keller-Sutter (Federal Finance Minister)] → 107 splits


  [negotiated end to the war / ceasefire agreement] → 5 splits


  [Victims of the Crans-Montana fire and their families] → 41 splits


...

  [US geopolitical and strategic interests in the Arctic] → clean


  [Zurich District Court (Bezirksgericht Zürich)] → 4 splits


  [arrest and prosecution of Nicolás Maduro] → clean


  [conviction of René Benko for creditor fraud (betrügerische K] → clean


  [residents of Birsfelden residential quarters] → clean

  Total variants to split out: 1,191

=== SPLIT INSPECTION ===

[European Union (EU institutions and member states)] — 168 splits
  SPLIT: NATO allies (US, European states, Canada)
  SPLIT: European NATO allies (Germany, Poland, Italy, United Kingdom)
  SPLIT: NATO member states (Europe and North Atlantic region)
  SPLIT: NATO (as institutional collective)
  SPLIT: NATO allies (Germany, Finland, Denmark, Sweden, Norway, UK, Netherlands, Canada)
  SPLIT: European NATO states (UK, France, Spain, Greece)
  SPLIT: Western allies (US, Europe)
  SPLIT: NATO member states (Germany, Estonia, Poland)
  SPLIT: European allies (NATO partners)
  SPLIT: NATO member states (Poland, broader alliance)
  SPLIT: Western allies (France, Britain, US)
  SPLIT: NATO allies (Denmark, France, Germany, Poland)
  SPLIT: NATO member states (Poland, Romania, Finland, Baltic states)
  SPLIT: NATO allies (Spain, Italy, France, Germany)
  SPLIT: NATO allies (

In [8]:
# sanity check

sanity_check(canonical_map, label_counts, n=50)

Top 50 normalized entities by number of variants:

  → European Union (EU institutions and member states) (21 occurrences)
      NATO allies (US, European states, Canada) (1)
      European NATO allies (Germany, Poland, Italy, United Kingdom) (1)
      NATO member states (Europe and North Atlantic region) (1)
      NATO (as institutional collective) (1)
      NATO allies (Germany, Finland, Denmark, Sweden, Norway, UK, Netherlands, Canada) (1)
      European states (Germany, France, UK, Poland, Italy, Finland, Norway, Denmark, Netherlands) (1)
      European NATO states (UK, France, Spain, Greece) (1)
      European Union (Commission and member states) (1)
      Baltic states (Estonia, Latvia) (1)
      Western allies (US, Europe) (1)
      NATO member states (Germany, Estonia, Poland) (1)
      European allies (NATO partners) (1)
      NATO member states (Poland, broader alliance) (1)
      Western allies (France, Britain, US) (1)
      NATO allies (Denmark, France, Germany, Poland) (1

In [9]:
# inspect what split_candidates contains for these two groups
for canonical, variants in split_candidates.items():
    if any(x in canonical for x in ["Ceasefire agreement and hostage", "Overthrow of the Islamic"]):
        print(f"\n[{canonical}] — {len(variants)} variants sent to LLM")
        for v in variants[:10]:
            print(f"  {v}")


[Ceasefire agreement and hostage release] — 111 variants sent to LLM
  ceasefire agreements in the Gaza conflict
  Ceasefire agreement with hostage release and Israeli withdrawal
  ceasefire agreement and hostage-prisoner exchange
  Ceasefire agreement and hostage release deal with Hamas
  Ceasefire and hostage exchange agreement
  Hamas agreement to ceasefire terms
  Ceasefire agreement with simultaneous hostage release
  Ceasefire between Iran and Israel
  Ceasefire agreement and prisoner exchange (Gaza deal)
  Gaza peace agreement with hostage-prisoner exchange and territorial settlement

[Overthrow of the Islamic Republic regime] — 61 variants sent to LLM
  Regime change in Venezuela / removal of Maduro from power
  Regime change in Iran / overthrow of Iranian leadership
  Overthrow of the Iranian Islamic Republic regime
  Overthrow of the Islamic Republic's political system
  Overthrow of the Iranian regime / regime change
  Regime change in Iran / overthrow of Islamic Republic l

In [10]:
# re-run split step on just these two problematic canonicals
import importlib
import lib.normalize_lib as normalize_lib
importlib.reload(normalize_lib)
from lib.normalize_lib import find_split_candidates, llm_split_canonicals, apply_splits

test_candidates = {
    k: v for k, v in split_candidates.items()
    if any(x in k for x in [
        "Ceasefire agreement and hostage",
        "Overthrow of the Islamic Republic"
    ])
}

test_splits = llm_split_canonicals(test_candidates, label_counts)

print("\n=== SPLIT INSPECTION ===")
for canonical, splits in test_splits.items():
    print(f"\n[{canonical}] — {len(splits)} splits")
    for s in splits[:20]:
        print(f"  SPLIT: {s}")

  [Ceasefire agreement and hostage release] → 18 splits


  [Overthrow of the Islamic Republic regime] → 14 splits

  Total variants to split out: 32

=== SPLIT INSPECTION ===

[Ceasefire agreement and hostage release] — 18 splits
  SPLIT: Ceasefire between Iran and Israel
  SPLIT: ceasefire agreement between Iran and Israel
  SPLIT: Ceasefire following Israel-Iran conflict
  SPLIT: Ceasefire agreement between Trump administration and Iran
  SPLIT: ceasefire and diplomatic settlement between Israel and Lebanon
  SPLIT: ceasefire between Israel and Iran
  SPLIT: ceasefire or peace agreement between Israel and Lebanon
  SPLIT: Ceasefire agreement between Israel and Syria
  SPLIT: Written ceasefire agreement between Iran and Israel
  SPLIT: Ceasefire between Israel and Hezbollah in Lebanon
  SPLIT: Ceasefire extension between Israel and Lebanon
  SPLIT: Ceasefire announcement between Iran and Israel
  SPLIT: Ceasefire agreement between Israel and Iran
  SPLIT: Ceasefire between Israel and Iran
  SPLIT: Ceasefire between Israel and Lebanon
  SPLI

In [11]:
# only apply after inspecting sanity check
if splits_by_canonical:
    canonical_map = apply_splits(canonical_map, splits_by_canonical)

  Split out 1,191 variants as singletons
  Unique canonicals after: 306,245


In [12]:
# group size diagnostic — run before LLM merging cell to decide on min_frequency cutoff

from collections import Counter
from sklearn.preprocessing import normalize as sk_normalize

label_to_idx  = {l: i for i, l in enumerate(labels)}
freq_map      = label_counts.set_index("label")["frequency"].to_dict()
raw_to_canon  = canonical_map.set_index("raw_label")["canonical_label"].to_dict()

# build canonical type map if not already built
canonical_type_map = build_canonical_type_map(extractions, canonical_map)

# group canonical labels by specific_type, count at different frequency thresholds
print("Group sizes by specific_type (canonical labels per group):\n")
print(f"{'specific_type':<45} {'all':>7} {'freq≥2':>7} {'freq≥3':>7} {'freq≥5':>7} {'freq≥10':>8}")
print("-" * 80)

type_to_canonicals = {}
for canon, stype in canonical_type_map.items():
    if canon in label_to_idx:
        type_to_canonicals.setdefault(stype, []).append(canon)

for stype, canons in sorted(type_to_canonicals.items(), key=lambda x: -len(x[1])):
    freqs = [freq_map.get(c, 0) for c in canons]
    n_all  = len(canons)
    n_ge2  = sum(f >= 2  for f in freqs)
    n_ge3  = sum(f >= 3  for f in freqs)
    n_ge5  = sum(f >= 5  for f in freqs)
    n_ge10 = sum(f >= 10 for f in freqs)
    print(f"{stype:<45} {n_all:>7,} {n_ge2:>7,} {n_ge3:>7,} {n_ge5:>7,} {n_ge10:>8,}")

print(f"\nNote: matrix size = n² entries. At n=20,000 → 400M entries (~3.2GB float32)")
print(f"Safe upper limit per group: ~15,000-20,000 labels without min_frequency cutoff")

Group sizes by specific_type (canonical labels per group):

specific_type                                     all  freq≥2  freq≥3  freq≥5  freq≥10
--------------------------------------------------------------------------------
policy_instrument                              37,202     706     185      53        8
affected_community                             30,018   2,412     882     383      133
institutional_framework                        20,519     327      79      22        2
foreign_executive                              16,162   1,595     848     435      211
libertarian                                    12,584     814     335     140       62
legal_instrument                               11,076     219      39      13        1
economic                                       10,731     168      47      16        4
corporation_firm                                9,635     675     289     123       49
communitarian                                   8,700     470     181      7

In [13]:
# LLM post-processing: iterative merging grouped by specific_type
# - candidate pairs found within each specific_type group separately
# - eliminates cross-type false positives structurally
# - better recall for infrequent entities within smaller type groups

import importlib
from config import normalize_config
importlib.reload(normalize_config)
cfg = normalize_config

import lib.normalize_lib as normalize_lib
importlib.reload(normalize_lib)
from lib.normalize_lib import (
    build_canonical_type_map, find_candidate_pairs_by_type,
    llm_adjudicate_pairs_by_type, apply_llm_merges_by_type,
)
from prompts.normalize_prompt import SYSTEM_PROMPT as NORMALIZE_PROMPT

print(f"Using similarity threshold: {cfg.LLM_POST_SIMILARITY}")
print(f"Using min_frequency: {cfg.LLM_POST_MIN_FREQUENCY}")

# build canonical → specific_type map
print("\nBuilding canonical type map...")
canonical_type_map = build_canonical_type_map(extractions, canonical_map)
print(f"  Types mapped for {len(canonical_type_map):,} canonical labels")

# inspect group sizes before running to check for oversized groups
from collections import Counter
type_counts = Counter(canonical_type_map.values())
print(f"\nCanonical labels per specific_type:")
for stype, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    print(f"  {stype}: {count:,}")

for round_num in range(1, cfg.LLM_POST_MAX_ROUNDS + 1):
    print(f"\n{'='*50}")
    print(f"Round {round_num}")
    print(f"{'='*50}")

    print("Finding candidate pairs by specific_type...")
    candidates_by_type = find_candidate_pairs_by_type(
        embeddings           = embeddings,
        labels               = labels,
        label_counts         = label_counts,
        canonical_map        = canonical_map,
        canonical_type_map   = canonical_type_map,
        similarity_threshold = cfg.LLM_POST_SIMILARITY,
        min_frequency        = cfg.LLM_POST_MIN_FREQUENCY,
    )

    total = sum(len(v) for v in candidates_by_type.values())
    if total == 0:
        print("No candidate pairs found — stopping.")
        break

    print("\nAdjudicating with LLM...")
    results_by_type = llm_adjudicate_pairs_by_type(
        candidates_by_type = candidates_by_type,
        system_prompt      = NORMALIZE_PROMPT,
        batch_size         = cfg.LLM_POST_BATCH_SIZE,
    )

    total_yes = sum(sum(r.values()) for r in results_by_type.values())
    if total_yes == 0:
        print("No merges confirmed — stopping.")
        break

    print("\nApplying merges...")
    canonical_map = apply_llm_merges_by_type(
        candidates_by_type = candidates_by_type,
        results_by_type    = results_by_type,
        canonical_map      = canonical_map,
        label_counts       = label_counts,
    )

    # rebuild type map to reflect updated canonicals
    canonical_type_map = build_canonical_type_map(extractions, canonical_map)
    print(f"Unique labels after round {round_num}: {canonical_map['canonical_label'].nunique():,}")

print(f"\nFinal unique labels: {canonical_map['canonical_label'].nunique():,}")

Using similarity threshold: 0.94
Using min_frequency: 2

Building canonical type map...
  Types mapped for 306,245 canonical labels

Canonical labels per specific_type:
  policy_instrument: 37,202
  affected_community: 30,018
  institutional_framework: 20,519
  foreign_executive: 16,162
  libertarian: 12,584
  legal_instrument: 11,076
  economic: 10,731
  corporation_firm: 9,635
  communitarian: 8,700
  political_party: 6,324
  individual_unaffiliated: 6,288
  geopolitical: 6,200
  executive: 5,869
  technological_instrument: 5,701
  regulatory_agency: 5,581
  communicative_instrument: 5,570
  authoritarian: 5,535
  cosmopolitan: 5,396
  market_liberal: 5,152
  cultural_historical: 4,971
  technocratic: 4,775
  redistributive: 4,492
  legislature: 4,275
  journalist_media_outlet: 4,059
  citizens_voters: 3,942
  economic_instrument: 3,776
  social_movement_initiative: 3,401
  security_enforcement: 2,975
  foreign_public_administration: 2,720
  environmental: 2,625
  public_administrati

  [academic_scientific] 3 pairs → YES: 3, NO: 0


  [affected_community] 3,933 pairs → YES: 1,389, NO: 2,544


  [authoritarian] 1,106 pairs → YES: 361, NO: 745


  [citizens_voters] 2,422 pairs → YES: 969, NO: 1,453


...

  [social_movement_initiative] 1 pairs → YES: 0, NO: 1


  [supranational_actor] 12 pairs → YES: 0, NO: 12


  [technocratic] 10 pairs → YES: 0, NO: 10


  [technological_instrument] 25 pairs → YES: 0, NO: 25


  [workers_employees] 66 pairs → YES: 0, NO: 66

  Total — YES: 3, NO: 1,239

Applying merges...
  Confirmed merges applied: 3
Unique labels after round 5: 302,099

Final unique labels: 302,099


In [14]:
# sanity check

sanity_check(canonical_map, label_counts, n=50)

Top 50 normalized entities by number of variants:

  → Russia (Putin administration) (95 occurrences)
      Russian military and state apparatus (14)
      Russian intelligence services (9)
      Russian state and military apparatus (9)
      Russian military forces and state apparatus (3)
      Russian military and state security apparatus (3)
      Russian state security apparatus (2)
      Russian military and security apparatus (2)
      Putin regime and Russian state security apparatus (2)
      Russian government and Russian intelligence services (1)
      Russian intelligence services and Russian state (1)
      Russian state apparatus and military (1)
      Russian state apparatus and Putin regime (1)
      Russian state authorities and security apparatus (1)
      Russian state apparatus (Putin regime) (1)
      Russian state intelligence services (1)
      Vladimir Putin and Russian state apparatus (1)
      Putin administration and state apparatus (1)
      Russian state sec

In [15]:
# singleton attachment: attach freq=1 labels to established canonicals
# runs after the main LLM loop — handles labels excluded by min_frequency=2

import importlib
import lib.normalize_lib as normalize_lib
importlib.reload(normalize_lib)
from lib.normalize_lib import (
    find_singleton_attachment_pairs_by_type,
    llm_adjudicate_pairs_by_type,
    apply_llm_merges_by_type,
    build_canonical_type_map,
)
from prompts.normalize_prompt import SYSTEM_PROMPT as NORMALIZE_PROMPT

print("Building canonical type map...")
canonical_type_map = build_canonical_type_map(extractions, canonical_map)

for round_num in range(1, cfg.LLM_POST_MAX_ROUNDS + 1):
    print(f"\n{'='*50}")
    print(f"Singleton attachment round {round_num}")
    print(f"{'='*50}")

    print("Finding singleton → canonical candidate pairs by specific_type...")
    candidates_by_type = find_singleton_attachment_pairs_by_type(
        embeddings           = embeddings,
        labels               = labels,
        label_counts         = label_counts,
        canonical_map        = canonical_map,
        canonical_type_map   = canonical_type_map,
        similarity_threshold = cfg.LLM_POST_SIMILARITY,
        anchor_min_frequency = 2,
    )

    total = sum(len(v) for v in candidates_by_type.values())
    if total == 0:
        print("No candidate pairs found — stopping.")
        break

    print("\nAdjudicating with LLM...")
    results_by_type = llm_adjudicate_pairs_by_type(
        candidates_by_type = candidates_by_type,
        system_prompt      = NORMALIZE_PROMPT,
        batch_size         = cfg.LLM_POST_BATCH_SIZE,
    )

    total_yes = sum(sum(r.values()) for r in results_by_type.values())
    if total_yes == 0:
        print("No merges confirmed — stopping.")
        break

    print("\nApplying merges...")
    canonical_map = apply_llm_merges_by_type(
        candidates_by_type = candidates_by_type,
        results_by_type    = results_by_type,
        canonical_map      = canonical_map,
        label_counts       = label_counts,
    )

    # rebuild type map — merged singletons now point to new canonicals
    canonical_type_map = build_canonical_type_map(extractions, canonical_map)
    print(f"Unique labels after singleton attachment round {round_num}: "
          f"{canonical_map['canonical_label'].nunique():,}")

print(f"\nFinal unique labels after singleton attachment: "
      f"{canonical_map['canonical_label'].nunique():,}")

Building canonical type map...

Singleton attachment round 1
Finding singleton → canonical candidate pairs by specific_type...
  [academic_scientific] 2,355 singletons x 64 anchors -> 19 candidate pairs
  [affected_community] 27,606 singletons x 1,753 anchors -> 2,468 candidate pairs
  [authoritarian] 5,107 singletons x 229 anchors -> 880 candidate pairs
  [citizens_voters] 3,402 singletons x 267 anchors -> 596 candidate pairs
  [civic_association] 1,957 singletons x 109 anchors -> 41 candidate pairs
  [civil_society] 440 singletons x 12 anchors -> 0 candidate pairs
  [civil_society / legal_professional] 5 singletons x 1 anchors -> 0 candidate pairs
  [civil_society / ngo_advocacy] 5 singletons x 1 anchors -> 0 candidate pairs
  [communicative_instrument] 5,516 singletons x 48 anchors -> 66 candidate pairs
  [communitarian] 8,230 singletons x 219 anchors -> 636 candidate pairs
  [consumers_households] 1,554 singletons x 123 anchors -> 144 candidate pairs
  [corporation_firm] 8,960 sing

  [academic_scientific] 19 pairs → YES: 16, NO: 3


  [affected_community] 2,468 pairs → YES: 1,209, NO: 1,259


  [authoritarian] 880 pairs → YES: 491, NO: 389


  [citizens_voters] 596 pairs → YES: 283, NO: 313


...

  [social_movement_initiative] 12 pairs → YES: 0, NO: 12


  [supranational_actor] 28 pairs → YES: 0, NO: 28


  [technocratic] 35 pairs → YES: 0, NO: 35


  [technological_instrument] 41 pairs → YES: 0, NO: 41


  [workers_employees] 68 pairs → YES: 0, NO: 68

  Total — YES: 43, NO: 4,864

Applying merges...
  Confirmed merges applied: 43
Unique labels after singleton attachment round 5: 291,631

Final unique labels after singleton attachment: 291,631


In [16]:
# sanity check

sanity_check(canonical_map, label_counts, n=50)

Top 50 normalized entities by number of variants:

  → Russia (Putin administration) (95 occurrences)
      Russian military and state apparatus (14)
      Russian intelligence services (9)
      Russian state and military apparatus (9)
      Russian military forces and state apparatus (3)
      Russian military and state security apparatus (3)
      Russian state security apparatus (2)
      Russian military and security apparatus (2)
      Putin regime and Russian state security apparatus (2)
      Russian government and Russian intelligence services (1)
      Russian intelligence services and Russian state (1)
      Russian state apparatus and military (1)
      Russian state apparatus and Putin regime (1)
      Russian state authorities and security apparatus (1)
      Russian state apparatus (Putin regime) (1)
      Russian state intelligence services (1)
      Vladimir Putin and Russian state apparatus (1)
      Putin administration and state apparatus (1)
      Russian state sec

In [19]:
# 1. how many raw labels and how many canonicals?
print(f"Raw labels:       {len(canonical_map):,}")
print(f"Unique canonicals: {canonical_map['canonical_label'].nunique():,}")

Raw labels:       353,162
Unique canonicals: 291,631


In [20]:
# 2. check a known singleton that should have been attached — 
#    e.g. one of the Staatsanwaltschaft variants
canonical_map[canonical_map["raw_label"].str.contains("Staatsanwalt", na=False)].head(5)

,raw_label,canonical_label,cluster_id,n_variants
4533,Italian judiciary / Staatsanwaltschaft,Italian State Prosecutor (Staatsanwaltschaft),334,14
4453,German law enforcement / Staatsanwaltschaft,German citizens and voters,334,144
4396,Italian prosecutors / Staatsanwaltschaft,Italian prosecutors / Staatsanwaltschaft,334,1
4490,Austrian prosecutors / Staatsanwaltschaft,Austrian State Prosecutor (Staatsanwaltschaft),334,10
4478,German prosecutors / Staatsanwaltschaft,German prosecutors / Staatsanwaltschaft,334,1


In [17]:
# group size diagnostic — run after completed LLM merging to check final group sizes

from collections import Counter
from sklearn.preprocessing import normalize as sk_normalize

label_to_idx  = {l: i for i, l in enumerate(labels)}
freq_map      = label_counts.set_index("label")["frequency"].to_dict()
raw_to_canon  = canonical_map.set_index("raw_label")["canonical_label"].to_dict()

# build canonical type map if not already built
canonical_type_map = build_canonical_type_map(extractions, canonical_map)

# group canonical labels by specific_type, count at different frequency thresholds
print("Group sizes by specific_type (canonical labels per group):\n")
print(f"{'specific_type':<45} {'all':>7} {'freq≥2':>7} {'freq≥3':>7} {'freq≥5':>7} {'freq≥10':>8}")
print("-" * 80)

type_to_canonicals = {}
for canon, stype in canonical_type_map.items():
    if canon in label_to_idx:
        type_to_canonicals.setdefault(stype, []).append(canon)

for stype, canons in sorted(type_to_canonicals.items(), key=lambda x: -len(x[1])):
    freqs = [freq_map.get(c, 0) for c in canons]
    n_all  = len(canons)
    n_ge2  = sum(f >= 2  for f in freqs)
    n_ge3  = sum(f >= 3  for f in freqs)
    n_ge5  = sum(f >= 5  for f in freqs)
    n_ge10 = sum(f >= 10 for f in freqs)
    print(f"{stype:<45} {n_all:>7,} {n_ge2:>7,} {n_ge3:>7,} {n_ge5:>7,} {n_ge10:>8,}")

print(f"\nNote: matrix size = n² entries. At n=20,000 → 400M entries (~3.2GB float32)")
print(f"Safe upper limit per group: ~15,000-20,000 labels without min_frequency cutoff")

Group sizes by specific_type (canonical labels per group):

specific_type                                     all  freq≥2  freq≥3  freq≥5  freq≥10
--------------------------------------------------------------------------------
policy_instrument                              36,570     638     154      46        7
affected_community                             27,980   1,753     533     210       71
institutional_framework                        20,236     270      66      21        2
foreign_executive                              14,495     873     449     228      114
libertarian                                    11,164     414     126      43       11
legal_instrument                               10,959     213      38      13        1
economic                                       10,619     134      36      16        4
corporation_firm                                9,375     604     267     118       49
communitarian                                   7,982     219      61      2

In [22]:
# debug: check high-frequency unmerged labels for missed normalization opportunities

# get all singleton canonical labels (not merged with anything)
singletons = canonical_map[canonical_map["n_variants"] == 1]["canonical_label"].values
singleton_counts = label_counts[label_counts["label"].isin(singletons)].sort_values("frequency", ascending=False)

print(f"Total unmerged labels: {len(singleton_counts):,}")
print(f"\nTop 500 most frequent unmerged labels:")
print(singleton_counts.head(500).to_string(index=False))

Total unmerged labels: 270,847

Top 500 most frequent unmerged labels:
                                                    label  frequency
                                             Donald Trump       1524
                                                  Ukraine        632
                                                   Russia        612
                                                     Iran        462
                                                   Israel        444
                                            United States        355
                                                    Hamas        317
                                           European Union        173
                                           Vladimir Putin        136
                                                Elon Musk        133
                                              Glückskette        126
                                                    China         96
                                

In [23]:
# apply normalization and save outputs

normalized = apply_normalization(extractions, canonical_map)

# filter extraction errors where model output a taxonomy value as the label
normalized = normalized[normalized["label"] != "market_liberal"]

save_outputs(normalized, canonical_map)

print(f"\nBefore normalization — unique labels: {extractions['label'].nunique():,}")
print(f"After normalization  — unique labels: {normalized['label'].nunique():,}")
print(f"\nTop 20 most frequent labels after normalization:")
print(normalized["label"].value_counts().head(20).to_string())

Saved normalized extractions → data/narrative_extractions_normalized.csv
Saved entity map → data/entity_normalization_map.csv

Before normalization — unique labels: 353,162
After normalization  — unique labels: 291,630

Top 20 most frequent labels after normalization:
label
United States (Trump administration)                         2261
Russia (Putin administration)                                1958
Swiss Federal Council (Bundesrat)                            1919
Swiss citizens and voters                                    1752
Rule of law and criminal justice authority                   1541
Donald Trump                                                 1524
Ukrainian civilian population                                1055
Communitarian sovereignty and national self-determination     998
Volodymyr Zelenskyy                                           702
Ukraine                                                       632
Gaza civilian population                                      619